In [ ]:
import os
import yaml
import json
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from anngeno import AnnGeno
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(num_cores)

In [ ]:
# Configuration and paths
corr_method = 'spearman'

config_path = f'/home/dnanexus/ukbgym/config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

maf = 1e-2 #config.get('maf', None)

pheno_path = '/home/dnanexus/data_dir/phenotypes/phenotypes190_missing20_unique2_int.parquet'
prs_path = '/home/dnanexus/data_dir/phenotypes/PRS190_EUR_missing20_unique2_int.parquet'
cov_path = '/home/dnanexus/data_dir/phenotypes/250709_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet'

anngeno_path = '/home/dnanexus/data_dir/dms_coding.ag'
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'
exp_data_path = '/home/dnanexus/data_dir/exp_data/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet'
save_path = None

# Read gene-trait combinations from dataframe
# Assuming you have a dataframe with columns: 'gene_id' and 'phenotype'
# Replace this with your actual dataframe loading
gene_trait_df = pl.DataFrame({
    'gene_id': ['ENSG00000106633'],  # Example gene IDs
    'phenotype': ['glycated_haemoglobin_hba1c_int']  # Example phenotypes
})



In [ ]:
# If you're loading from a file, use something like:
# gene_trait_df = pl.read_parquet('path_to_your_gene_trait_combinations.parquet')
# or
# gene_trait_df = pl.read_csv('path_to_your_gene_trait_combinations.csv')

def process_phenotypes_and_prs(phenotypes_list):
    """Process all phenotypes and PRS data at once"""
    
    # Read all required phenotypes
    phenos = pl.read_parquet(pheno_path).rename({'IID':'individual'}).select(['individual'] + phenotypes_list).drop_nulls()
    
    # Read all required PRS
    prs_cols = [f'{pheno}_prs' for pheno in phenotypes_list]
    prs = pl.read_parquet(prs_path).rename({'IID':'individual'}).select(['individual'] + prs_cols).drop_nulls()
    
    # Get covariates
    cov_list = config.get("covariates")
    cov_df = pl.read_parquet(cov_path).rename({
        'sample':'individual'
    }).select(
        ['individual'] + cov_list
    ).with_columns(
        pl.col('individual').cast(pl.Int64).alias('individual')
    )
    
    # Join all data
    all_df = phenos.join(prs, on='individual', how='inner').join(cov_df, on='individual', how='inner')
    all_pd = all_df.to_pandas()
    
    # Process each phenotype separately to get residuals
    residuals_list = []
    
    for phenotype in phenotypes_list:
        try:
            # Get columns for this phenotype (phenotype + its PRS + covariates)
            pheno_cols = [phenotype, f'{phenotype}_prs'] + cov_list
            temp_df = all_pd[['individual'] + pheno_cols].dropna()
            
            if len(temp_df) == 0:
                print(f"No data available for phenotype: {phenotype}")
                continue
            
            y = temp_df[phenotype]
            X = temp_df.drop(columns=[phenotype, 'individual'])
            X = sm.add_constant(X)  # Add constant term for intercept
            
            # Fit model and get residuals
            model = sm.OLS(y, X).fit()
            residuals = pd.Series(model.resid, index=temp_df.index, name=f'{phenotype}_residual')
            
            # Create dataframe with individual IDs and residuals
            pheno_residuals = pd.concat([temp_df[['individual']], residuals], axis=1)
            residuals_list.append(pheno_residuals)
            
        except Exception as e:
            print(f"Error processing phenotype {phenotype}: {e}")
            continue
    
    return residuals_list

def process_gene_genotypes(gene_id, ag):
    """Extract genotypes for a specific gene"""
    try:
        reg_dict = ag.get_region(gene_id)
        geno = reg_dict['genotypes']
        anno_df = reg_dict['annotations']
        samples = ag.samples
        
        # Find heterozygous genotypes (genotype == 1)
        rows, cols = np.where(geno == 1)
        het = pl.DataFrame({
            'id': np.array(anno_df['id'])[rows],
            'individual': np.array(samples)[cols],
            'genotype': 1
        })
        
        # Find homozygous genotypes (genotype == 2)
        rows, cols = np.where(geno == 2)
        hom = pl.DataFrame({
            'id': np.array(anno_df['id'])[rows],
            'individual': np.array(samples)[cols],
            'genotype': 2
        })
        
        geno_melt = pl.concat([het, hom]).with_columns(
            pl.lit(gene_id).alias('region')
        )
        
        return geno_melt, anno_df
        
    except Exception as e:
        print(f"Error processing genotypes for gene {gene_id}: {e}")
        return None, None

def bootstrap_samples(gpa_df: pl.DataFrame, n_bootstraps: int, seed: int = None) -> pl.DataFrame:
    """Bootstrap samples for correlation analysis"""
    if seed is not None:
        np.random.seed(seed)

    # Get unique individuals
    unique_ids = gpa_df.select('individual').unique().to_series().to_list()
    n_ids = len(unique_ids)

    # Prepare all bootstrap samples
    sampled_ids = np.random.choice(unique_ids, size=(n_bootstraps, n_ids), replace=True)

    # Flatten and make DataFrame with bootstrap_id
    boot_id_col = np.repeat(np.arange(n_bootstraps), n_ids)
    sampled_flat = pl.LazyFrame({
        'bootstrap_id': boot_id_col,
        'individual': sampled_ids.ravel()
    })

    # Lazy join to replicate rows
    boot_df = sampled_flat.join(gpa_df.lazy(), on='individual', how='left')

    # Group by bootstrap_id + original grouping columns
    result = (
        boot_df
        .group_by(['bootstrap_id', 'id', 'phenotype', 'region', 'annotation'])
        .agg([
            pl.len().alias('n_individuals'),
            pl.col('pheno_value').mean().alias('mean_pheno_value'),
            pl.col('score').mean().alias('mean_score'),
        ])
        .collect()
    )

    return result

In [ ]:
# Main processing pipeline
def main():
    # Get unique phenotypes for batch processing
    unique_phenotypes = gene_trait_df['phenotype'].unique().to_list()
    
    # Process all phenotypes at once to get residuals
    print("Processing phenotypes and PRS...")
    residuals_list = process_phenotypes_and_prs(unique_phenotypes)
    
    # Combine all residuals into one dataframe
    if not residuals_list:
        print("No valid phenotype data found")
        return
    
    # Convert to polars and melt
    all_residuals_dfs = []
    for residual_df in residuals_list:
        p_wide = pl.DataFrame(residual_df).with_columns(
            pl.col('individual').cast(pl.String).alias('individual')
        )
        
        pheno_cols = [col for col in p_wide.columns if col.endswith('_residual')]
        
        pdf = p_wide.unpivot(
            index=['individual'], 
            on=pheno_cols, 
            variable_name='phenotype',
            value_name='pheno_value'
        ).with_columns(
            pl.col('phenotype').str.replace('_residual', '').alias('phenotype')
        )
        
        all_residuals_dfs.append(pdf)
    
    combined_pdf = pl.concat(all_residuals_dfs) if len(all_residuals_dfs) > 1 else all_residuals_dfs[0]
    
    # Initialize AnnGeno
    print("Loading AnnGeno...")
    ag = AnnGeno(anngeno_path, mode='r', low_mem=True)
    
    # Filter variants by MAF
    if maf:
        variants_to_keep = ag.annotations.filter((pl.col('af_ukb') < maf)).select('id').collect()['id']
        ag.subset_variants(set(variants_to_keep))
    
    # Get EUR samples if specified
    eur_samples = None
    if eur_samples_path:
        eur_samples = pl.read_csv(eur_samples_path).with_columns(
            pl.col("eid").cast(pl.Utf8)
        )
    
    # Get DMS scores
    print("Loading DMS scores...")
    unique_genes = gene_trait_df['gene_id'].unique().to_list()
    dms_df = pl.read_parquet(exp_data_path).filter(pl.col('gene_id').is_in(unique_genes))
    
    # Process DMS scores
    dms_wide = dms_df.with_columns(
        pl.when(pl.col('file_name') == 'HXK4_HUMAN_Gersing_2022_activity')
          .then(pl.lit('dms_activity'))
          .when(pl.col('file_name') == 'HXK4_HUMAN_Gersing_2023_abundance')
          .then(pl.lit('dms_abundance'))
          .otherwise(None)
          .alias('score_type')
    ).pivot(
        index='mutant',
        on='score_type',
        values='dms_score'
    )
    
    # Get annotation categories
    all_annotation_list = []
    rare_variant_annotations_dict = config.get('rare_variant_annotations')
    if rare_variant_annotations_dict:
        for category in rare_variant_annotations_dict.values():
            all_annotation_list.extend(category)
    
    # Process each gene-trait combination
    all_results = []
    all_boot_results = []
    
    for row in tqdm(gene_trait_df.iter_rows(named=True), total=len(gene_trait_df)):
        gene_id = row['gene_id']
        phenotype = row['phenotype']
        
        print(f"Processing {gene_id} - {phenotype}")
        
        # Get genotypes for this gene
        geno_melt, anno_df = process_gene_genotypes(gene_id, ag)
        
        if geno_melt is None:
            continue
        
        # Filter phenotype data for this specific phenotype
        pheno_data = combined_pdf.filter(pl.col('phenotype') == phenotype)
        
        if len(pheno_data) == 0:
            print(f"No phenotype data for {phenotype}")
            continue
        
        # Join genotypes and phenotypes
        gp_df = geno_melt.join(pheno_data, on='individual')
        
        # Filter for EUR ancestry if specified
        if eur_samples is not None:
            gp_df = gp_df.filter(pl.col('individual').is_in(eur_samples['eid'].to_list()))
        
        # Remove homozygous variant carriers (adjust as needed)
        gp_df = gp_df.filter(pl.col('genotype') == 1)
        
        if len(gp_df) == 0:
            print(f"No valid genotype-phenotype data for {gene_id} - {phenotype}")
            continue
        
        # Process annotations
        try:
            anno_melt = anno_df.join(dms_wide, on='mutant', how='inner')
            available_annotations = list(set(all_annotation_list).intersection(set(anno_melt.columns)))
            
            if not available_annotations:
                print(f"No valid annotations for {gene_id}")
                continue
            
            anno_melt = anno_melt.unpivot(
                index=['id', 'region', 'af_ukb'],
                on=available_annotations,
                variable_name='annotation',
                value_name='score'
            )
            
            # Join with genotype-phenotype data
            gpa_df = gp_df.join(anno_melt, on='id', how='inner')
            
            if len(gpa_df) == 0:
                print(f"No data after joining annotations for {gene_id} - {phenotype}")
                continue
            
            # Compute correlations
            plot_df = gpa_df.group_by(['id', 'phenotype', 'region', 'annotation']).agg(
                pl.len().alias('n_individuals'),
                pl.col('pheno_value').mean().alias('mean_pheno_value'),
                pl.col('score').mean().alias('mean_score'),
            ).drop_nulls(subset=['mean_pheno_value', 'mean_score'])
            
            if len(plot_df) == 0:
                continue
            
            corr_df = (
                plot_df
                .group_by(['annotation', 'phenotype', 'region'])
                .agg([
                    pl.corr('mean_score', 'mean_pheno_value', method=corr_method).alias('corr')
                ])
                .with_columns([
                    pl.col('corr').abs().alias('abs_corr')
                ])
                .filter(pl.col('abs_corr').is_not_null())
                .sort('abs_corr', descending=True)
                .with_columns([
                    pl.lit(gene_id).alias('gene_id'),
                    pl.lit(phenotype).alias('phenotype_name')
                ])
            )
            
            all_results.append(corr_df)
            
            # Bootstrap analysis
            print(f"Running bootstrap for {gene_id} - {phenotype}")
            boot_df = bootstrap_samples(gpa_df, n_bootstraps=1000, seed=42)
            
            boot_corr_df = (
                boot_df
                .group_by(['phenotype', 'region', 'annotation', 'bootstrap_id'])
                .agg([
                    pl.corr('mean_score', 'mean_pheno_value', method=corr_method).alias('corr')
                ])
                .with_columns([
                    pl.col('corr').abs().alias('abs_corr')
                ])
                .sort('abs_corr', descending=True)
                .drop_nans()
                .with_columns([
                    pl.lit(gene_id).alias('gene_id'),
                    pl.lit(phenotype).alias('phenotype_name')
                ])
            )
            
            all_boot_results.append(boot_corr_df)
            
        except Exception as e:
            print(f"Error processing annotations for {gene_id} - {phenotype}: {e}")
            continue
    
    # Combine all results
    if all_results:
        final_corr_df = pl.concat(all_results)
        print(f"Final correlation results shape: {final_corr_df.shape}")
        
        # Save results
        if save_path is not None:
            final_corr_df.write_parquet(f'{save_path}/multi_gene_trait_correlations.parquet')

    if all_boot_results:
        final_boot_df = pl.concat(all_boot_results)
        print(f"Final bootstrap results shape: {final_boot_df.shape}")
        
        # Save bootstrap results
        if save_path is not None:
            final_boot_df.write_parquet(f'{save_path}/multi_gene_trait_bootstrap_correlations.parquet')

    return final_corr_df if all_results else None, final_boot_df if all_boot_results else None

In [ ]:
corr_results, boot_results = main()

if corr_results is not None:
    print("\nTop correlations across all gene-trait pairs:")
    # print(corr_results.sort('abs_corr', descending=True).head(10))

boot_results

In [ ]:
from plotnine import *

summary_df = (
    boot_results
    .group_by("annotation")
    .agg([
        pl.col("abs_corr").mean().alias("mean_abs_corr"),
        pl.col("abs_corr").std().alias("se_abs_corr"),
        pl.col("abs_corr").quantile(0.025).alias("quant_low"),
        pl.col("abs_corr").quantile(0.975).alias("quant_high"),
    ])
    .with_columns([
        (pl.col("mean_abs_corr") - 1.96*pl.col("se_abs_corr")).alias("ci_low"),
        (pl.col("mean_abs_corr") + 1.96*pl.col("se_abs_corr")).alias("ci_high")
    ])
    .to_pandas()
)

# Ordering by mean correlation
summary_df['annotation'] = pd.Categorical(
    summary_df['annotation'],
    categories=summary_df.sort_values('mean_abs_corr', ascending=False)['annotation'],
    ordered=True
)

summary_df['color_dms'] = summary_df['annotation'].str.startswith('dms_').map({True: 'DMS', False: 'Other'})

# Plot
(
    ggplot(summary_df, aes(x='annotation', y='mean_abs_corr', fill='color_dms')) +
    geom_col(alpha=0.8) +
    # geom_errorbar(aes(ymin='ci_low', ymax='ci_high'), width=0.2) +
    geom_errorbar(aes(ymin='quant_low', ymax='quant_high'), width=0.2) +
    coord_flip() +
    theme_538() +
    labs(
        x='Annotation',
        y='Mean absolute correlation (95% CI)'
    ) +
    theme(
        figure_size=(8, 3),
        legend_position='none'
    )
)